# 07 — Lifecycle Environmental Impact Assessment

**Objective:** Estimate the environmental footprint of AED deployment
across different scaling scenarios using a stock-and-flow lifecycle model.

**Approach:**
- A modular `EnvironmentTracer` simulates 10-year material demand and CO₂ emissions
- Component replacement cycles (batteries, structural brackets) drive periodic demand spikes
- Coefficients are configurable placeholders, designed to be replaced with
  verified LCA databases (e.g., ecoinvent, GREET) in future work

> **Note:** The lifecycle coefficients used here are illustrative estimates.
> Rigorous application would require validated data from manufacturer specifications
> or published LCA studies.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('.')
OUT_DIR = PROJECT_ROOT / 'mda_project' / 'data' / 'output'

scen = pd.read_csv(PROJECT_ROOT / 'mda_project' / 'data' / 'output' / 'figures' / 'table_05_scenarios.csv')
# Normalise column names to lowercase_underscore if needed
scen.columns = [c.lower().replace(' ', '_').replace('<', '').replace('[', '').replace(']', '').replace('/', '_').replace('%', 'pct') for c in scen.columns]
print(f'Scenarios loaded: {len(scen)}')
print(scen.head())

## 7.1 Lifecycle Simulation Model

In [ ]:
class EnvironmentTracer:
    """Stock-and-flow lifecycle model for infrastructure material demand and emissions.
    
    Parameters
    ----------
    elements_config : dict
        Component specifications with lifecycle duration, mass, and CO2 intensity.
        Designed as a pluggable interface for validated LCA databases.
    """
    def __init__(self, elements_config):
        self.config = elements_config

    def simulate_10_years(self, n_nodes):
        records = []
        for year in range(1, 11):
            for component, params in self.config.items():
                is_replacement = 1 if (year == 1 or year % params['life_y'] == 0) else 0
                records.append({
                    'year': year,
                    'component': component,
                    'demand_kg': n_nodes * params['mass_kg'] * is_replacement,
                    'co2_kg': n_nodes * params['co2_kg'] * is_replacement,
                })
        return pd.DataFrame(records)


# Placeholder coefficients (to be replaced with validated LCA data)
COMPONENT_CONFIG = {
    'battery_pack':        {'life_y': 4, 'mass_kg': 0.5, 'co2_kg': 15.0},
    'structural_bracket':  {'life_y': 8, 'mass_kg': 3.0, 'co2_kg': 50.0},
}

tracer = EnvironmentTracer(COMPONENT_CONFIG)

## 7.2 Run Simulation Across All Scenarios

In [ ]:
results = []
for _, s in scen.iterrows():
    sim = tracer.simulate_10_years(int(s['new_aeds']))
    agg = sim.groupby('component')[['demand_kg', 'co2_kg']].sum()

    results.append({
        'Scenario': s['scenario'],
        'New AEDs': int(s['new_aeds']),
        'Coverage <1km': s['coverage_1km'],
        'Total Cost [EUR]': s['total_cost_eur'],
        'Battery Demand 10y [kg]': agg.loc['battery_pack', 'demand_kg'],
        'Total CO2 10y [kg]': agg['co2_kg'].sum() + s['new_aeds'] * 4 * 10,
    })

mfa_df = pd.DataFrame(results)
print(mfa_df)

## 7.3 Cumulative Emissions Visualization

In [ ]:
# Detailed year-by-year breakdown for S30 scenario
example_sim = tracer.simulate_10_years(30)
pivot = example_sim.pivot_table(index='year', columns='component', values='co2_kg', aggfunc='sum').fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stacked bar: annual CO2 by component
pivot.plot(kind='bar', stacked=True, ax=axes[0], color=['#3498db', '#e74c3c'], edgecolor='white')
axes[0].set_title('S30: Annual CO2 Emissions by Component', fontweight='bold')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('CO2 [kgCO2e]')
axes[0].legend(title='Component')

# Total CO2 across scenarios
axes[1].bar(mfa_df['Scenario'], mfa_df['Total CO2 10y [kg]'], color='#2ecc71', edgecolor='white')
axes[1].set_title('10-Year Cumulative CO2 by Scenario', fontweight='bold')
axes[1].set_xlabel('Scenario')
axes[1].set_ylabel('Total CO2 [kgCO2e]')

plt.tight_layout()
plt.show()

## Summary

The lifecycle model provides a structured framework for assessing environmental costs
alongside operational benefits. The modular `EnvironmentTracer` design allows seamless
integration with validated LCA databases in future work.

Combined with the cost and equity metrics from **Notebook 05**, this enables
comprehensive multi-criteria decision support for AED network planning.